# Tome 4, Chapter 5: The Gamma and Beta Functions (Python companion)

Exercises first, solutions at the end. We check the identities of the chapter to fifty digits, watch Gauss's product converge, build the Lanczos approximation that most numerical libraries use for $\Gamma$, learn why programs compute $\ln\Gamma$ rather than $\Gamma$, and measure how empty high-dimensional cubes are. Requirements: numpy and mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P5.1 (★) Fifty digits.** Check the reflection formula (Theorem 5.13), the duplication formula (Theorem 5.14), Gauss's multiplication formula for $m=3$ (Exercise 5.6), Theorem 5.11, Proposition 5.12, Exercise 5.5 and Exercise 5.7 to $50$ digits. Why does `quad` struggle with $B(\frac13,\frac27)$, and how do you fix it?

**P5.2 (★) Gauss's product.** How fast does $\frac{n!\,n^x}{x(x+1)\cdots(x+n)}$ converge to $\Gamma(x)$ (Theorem 5.6)? Guess the leading term of the relative error from the data.

**P5.3 (★★) CS application: the Lanczos approximation.** In 1964 Lanczos found, for a parameter $g$, coefficients $p_k$ with
$$\Gamma(z+1)\approx\sqrt{2\pi}\,\Bigl(z+g+\tfrac12\Bigr)^{z+\frac12}e^{-(z+g+\frac12)}\Bigl(p_0+\sum_{k=1}^{8}\frac{p_k}{z+k}\Bigr),$$
a Stirling-like main factor times a rational correction. Implement it with the standard coefficients for $g=7$, use the reflection formula for $z<\frac12$, and measure the worst relative error on $[-0.02,100]$.

**P5.4 (★) Logarithms, not values.** $\Gamma(172)$ overflows a double. Compute the number of digits of $\binom{10^6}{5\cdot10^5}$ with `math.lgamma`, and compare with the estimate $2^n\sqrt{2/(\pi n)}$. Then measure the error of Stirling's series for $\ln\Gamma(x)$ with $0$ to $4$ correction terms.

**P5.5 (★) Why log-convexity?** Check numerically that $\ln\Gamma$ is convex while $\ln\Gamma(x)+\ln(1+0.3\sin2\pi x)$ (the think box before Figure 5.2) is not.

**P5.6 (★★) Empty cubes.** Sample $10^6$ uniform points in $[-1,1]^n$ and count those in the unit ball, for $n=2,3,5,10,15,20$; compare with $V_n/2^n$ from Theorem 5.15. What fraction of the ball lies within $0.05$ of its surface?

## Solutions

### P5.1

In [ ]:
from mpmath import mp, gamma, beta, pi, sqrt, sin, quad, euler, diff, mpf, nstr, log, loggamma
mp.dps = 50
x, y = mpf(1) / 3, mpf(2) / 7
checks = [
    ("reflection  Gamma(x)Gamma(1-x) = pi/sin(pi x)", gamma(x) * gamma(1 - x), pi / sin(pi * x)),
    ("duplication                               ", gamma(x) * gamma(x + mpf(1) / 2), 2**(1 - 2*x) * sqrt(pi) * gamma(2*x)),
    ("multiplication, m = 3                     ", gamma(x) * gamma(x + mpf(1)/3) * gamma(x + mpf(2)/3),
     (2*pi) * 3**(mpf(1)/2 - 3*x) * gamma(3*x)),
    ("Beta = Gamma Gamma / Gamma                ", # split at 1/2; on [0, 1/2] substitute t = v^(1/x), on [1/2, 1] substitute 1 - t = v^(1/y)
     quad(lambda v: (1 - v**(1/x))**(y-1), [0, mpf(2)**(-x)]) / x
     + quad(lambda v: (1 - v**(1/y))**(x-1), [0, mpf(2)**(-y)]) / y,
     gamma(x) * gamma(y) / gamma(x + y)),
    ("Gamma(1/2) = sqrt(pi)                     ", gamma(mpf(1) / 2), sqrt(pi)),
    ("lemniscate integral                       ", quad(lambda w: 2 / sqrt((2 - w**2) * (1 + (1 - w**2)**2)), [0, 1]),   # r = 1 - w^2
     gamma(mpf(1) / 4)**2 / (4 * sqrt(2 * pi))),
    ("Gamma'(1) = -gamma                        ", diff(gamma, 1), -euler),
]
naive = quad(lambda t: t**(x-1) * (1-t)**(y-1), [0, 1])
print("Beta by quad on [0, 1] directly: difference", nstr(abs(naive - gamma(x) * gamma(y) / gamma(x + y)), 3))
for name, a, b in checks:
    print(f"{name}  difference {nstr(abs(a - b), 3)}")

All identities hold to the last digit. The Beta integral is the instructive case: tanh-sinh quadrature handles mild endpoint singularities such as $\ln t$, but $t^{-2/3}$ and $(1-t)^{-5/7}$ cost it most of its digits (the first line: only $15$). The cure is analytic, not numerical: on $[0,\frac12]$ substitute $t=v^{1/x}$, which turns $t^{x-1}\dd t$ into $\frac1x\dd v$ and the integrand into a smooth function. The same idea, $r=1-w^2$, removes the square-root singularity of the lemniscate integral at $r=1$. Knowing where an integrand is singular, and removing the singularity by a change of variables, is most of the art of numerical integration.

### P5.2

In [ ]:
mp.dps = 30
def gauss_product(x, n):
    p = mpf(1)
    for k in range(n + 1):
        p *= (x + k)
    from mpmath import factorial
    return factorial(n) * mpf(n)**x / p
x = mpf(1) / 2
for n in (10, 100, 1000, 10000):
    err = gauss_product(x, n) / gamma(x) - 1
    print(f"n = {n:5d}: relative error {nstr(err, 4):>11}   -x(x+1)/(2n) = {nstr(-x*(x+1)/(2*n), 4)}")

The relative error is about $-\frac{x(x+1)}{2n}$: Gauss's product converges like $\frac1n$, too slowly for computation. Expanding $\ln$ of the ratio of the product to $\Gamma(x)$ with the Euler–Maclaurin formula of Chapter 11 gives exactly this term.

### P5.3

In [ ]:
import math
G, P = 7, [0.99999999999980993, 676.5203681218851, -1259.1392167224028, 771.32342877765313,
           -176.61502916214059, 12.507343278686905, -0.13857109526572012,
           9.9843695780195716e-6, 1.5056327351493116e-7]

def lanczos_gamma(z):
    if z < 0.5:                                    # reflection formula
        return math.pi / (math.sin(math.pi * z) * lanczos_gamma(1 - z))
    z -= 1
    a = P[0] + sum(P[i] / (z + i) for i in range(1, len(P)))
    t = z + G + 0.5
    return math.sqrt(2 * math.pi) * t**(z + 0.5) * math.exp(-t) * a

worst = 0
for k in range(1, 2000):
    z = k / 20 - 0.017                              # avoid the poles exactly
    rel = abs(lanczos_gamma(z) / float(gamma(z)) - 1)
    worst = max(worst, rel)
print(f"Lanczos, 9 coefficients: worst relative error on [-0.02, 100] = {worst:.1e}")
print("Gamma(4.5):", lanczos_gamma(4.5), " exact", nstr(gamma(4.5), 17))
print("Gamma(-2.5):", lanczos_gamma(-2.5), " exact", nstr(gamma(-2.5), 17))

Nine coefficients give a relative error below $10^{-13}$ on the whole range, close to double precision, at the cost of one power, one exponential and nine divisions. The idea: $\Gamma(z+1)e^{z+g+1/2}(z+g+\frac12)^{-z-1/2}$ is a very smooth function of $z$ for a well-chosen $g$, so a short rational approximation captures it. Variants of this formula are behind `math.gamma` in many languages; the reflection formula handles the left half-line, poles included.

### P5.4

In [ ]:
print("math.gamma(171) =", math.gamma(171))
try:
    math.gamma(172)
except OverflowError as e:
    print("math.gamma(172):", e)
n = 10**6
lnC = math.lgamma(n + 1) - 2 * math.lgamma(n // 2 + 1)
print(f"ln C(10^6, 5*10^5) = {lnC:.6f}, so C has {math.floor(lnC / math.log(10)) + 1} digits")
print("compare with ln(2^n sqrt(2/(pi n))):", n * math.log(2) + 0.5 * math.log(2 / (math.pi * n)))

def stirling_series(x, terms):
    c = [mpf(1)/12, -mpf(1)/360, mpf(1)/1260, -mpf(1)/1680]
    s = (x - mpf(1)/2) * log(x) - x + log(2 * pi) / 2
    for j in range(terms):
        s += c[j] / x**(2*j + 1)
    return s
for x in (2, 5, 10):
    errs = [nstr(abs(stirling_series(mpf(x), m) - loggamma(x)), 2) for m in range(0, 5)]
    print(f"x = {x:2d}: error with 0..4 correction terms: {errs}")

Doubles stop at about $1.8\times10^{308}$, and $171!\approx1.2\times10^{309}$. Numerical code therefore works with $\ln\Gamma$ (`lgamma`), for binomial coefficients, likelihoods of Poisson and Gamma distributions (Tome 3) and Beta priors in Bayesian statistics. The central binomial coefficient has $301\,027$ digits, and Stirling's estimate agrees with `lgamma` to all printed digits. Stirling's series improves with each term at $x=5$ or $10$, but at $x=2$ the gains slow down quickly: the series diverges for every $x$, and Chapter 11 explains where to stop.

### P5.5

In [ ]:
import numpy as np
f = lambda x: math.lgamma(x) + math.log(1 + 0.3 * math.sin(2 * math.pi * x))
h = 1e-3
xs = np.arange(0.3, 5, 0.01)
bad = [x for x in xs if f(x + h) - 2 * f(x) + f(x - h) < 0]
print(f"impostor: second differences negative on {len(bad)} of {len(xs)} sample points, e.g. x = {bad[0]:.2f}")
good = [x for x in xs if math.lgamma(x + h) - 2 * math.lgamma(x) + math.lgamma(x - h) < 0]
print("ln Gamma: negative second differences at", len(good), "points")

The impostor satisfies $f(1)=1$ and $f(x+1)=xf(x)$, but its logarithm has negative second differences about half the time: the hypothesis that singles out $\Gamma$ in Theorem 5.6 is log-convexity. Plain convexity of $f$ is not enough: with $0.01$ instead of $0.3$, the impostor is convex (check its second differences) but its logarithm is still not, which is why the theorem asks for the logarithm.

### P5.6

In [ ]:
rng = np.random.default_rng(5)
for n in (2, 3, 5, 10, 15, 20):
    V = math.pi**(n / 2) / math.gamma(n / 2 + 1)
    N = 10**6
    pts = rng.uniform(-1, 1, size=(N, n))
    inside = (np.einsum('ij,ij->i', pts, pts) <= 1).mean()
    print(f"n = {n:2d}: V_n = {V:8.5f}  fraction of cube = {V / 2**n:.2e}  Monte Carlo {inside:.2e}")
for n in (3, 10, 100):
    print(f"n = {n:3d}: fraction of the ball within 0.05 of the surface = {1 - 0.95**n:.3f}")

Up to $n=10$ the Monte Carlo counts match $V_n/2^n$. At $n=15$ only about $12$ of the $10^6$ points are expected inside the ball, so the estimate is mostly noise, and at $n=20$ none at all: rejection sampling (draw in the cube, keep what falls in the ball) needs about $4\times10^7$ draws per accepted point. In dimension $100$, $99.4\%$ of the ball's volume lies within $0.05$ of its surface. To sample uniformly in a high-dimensional ball one draws a Gaussian vector, normalises it, and rescales the radius by $U^{1/n}$ with $U$ uniform: the Gaussian's rotation invariance (Tome 3) replaces rejection.